# 3D Drone Video Reconstruction — Colab GPU Temporal Worker

This notebook bootstraps and runs the Python Temporal CV worker for single-pass 3D drone reconstruction on a Google Colab T4 GPU runtime.

### Prerequisites
1. **Runtime:** Set runtime to **GPU (T4)** via `Runtime -> Change runtime type -> T4 GPU`.
2. **Secrets:** In the left sidebar **Secrets (🔑)** tab, add:
   - `TEMPORAL_HOST` (e.g. `temporal.yourdomain.com:443` or reverse tunnel host)
   - `PRESIGN_ENDPOINT` (e.g. `https://api.yourdomain.com/v1/storage/presign`)
   - `MINIO_ENDPOINT` (e.g. `s3.yourdomain.com`)
   - `MINIO_ACCESS_KEY` / `MINIO_SECRET_KEY` (if using static auth mode)
   - `POSTGRES_DSN` (Postgres connection string for metadata updates)
   - `GH_TOKEN` (optional, for private repository clones)


## Cell 1: Hardware & GPU Check

In [ ]:
#@title 1. Verify GPU Runtime
import torch
import sys

print(f"Python Version: {sys.version}")
if not torch.cuda.is_available():
    raise SystemError("❌ GPU is not available! Go to Runtime -> Change runtime type -> select T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ Attached GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
!nvidia-smi


## Cell 2: Clone or Update Codebase (Idempotent)

In [ ]:
#@title 2. Repository Setup
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/saalineo/single-pass-drone-video-3d-reconstruction.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
TARGET_DIR = "/content/single-pass-drone-video-3d-reconstruction" #@param {type:"string"}

try:
    from google.colab import userdata
    gh_token = userdata.get("GH_TOKEN")
    if gh_token and "https://" in REPO_URL:
        REPO_URL = REPO_URL.replace("https://", f"https://{gh_token}@")
except Exception:
    pass

if not os.path.exists(TARGET_DIR):
    print(f"Cloning {REPO_URL} into {TARGET_DIR}...")
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {TARGET_DIR}
else:
    print(f"Target directory {TARGET_DIR} exists. Updating branch {BRANCH}...")
    !cd {TARGET_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

os.chdir(os.path.join(TARGET_DIR, "workers/cv-python"))
print(f"✅ Current Working Directory: {os.getcwd()}")


## Cell 3: Install Dependencies & Download Model Checkpoints

In [ ]:
#@title 3. Install System / Python Dependencies & Download Checkpoints
import os
import sys
import urllib.request
from pathlib import Path

print("Installing system packages (FFmpeg, libgl1, boost, ceres)... ")
!apt-get update -qq && apt-get install -y -qq ffmpeg libgl1 libglew2.2 libboost-all-dev libfreeimage-dev > /dev/null

print("Installing core Python worker packages...")
!pip install -q boto3 ffmpeg-python opencv-python-headless pycolmap pydantic pyproj structlog temporalio transformers laspy[lazrs] rasterio trimesh plyfile open3d asyncpg kornia

print("Installing SAM2...")
!pip install -q "git+https://github.com/facebookresearch/sam2.git@2.1"

print("Installing gsplat (targeting Turing CC 7.5 for Colab T4)...")
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5;8.0;8.6"
!pip install -q gsplat==1.3.0

# Install local worker package in editable mode
!pip install -q -e .

print("Setting up model checkpoints directory at /models...")
os.makedirs("/models/sam2", exist_ok=True)
os.makedirs("/models/grounding-dino-tiny", exist_ok=True)

# 1. Download SAM 2.1 Hiera Large checkpoint if missing
sam2_ckpt = Path("/models/sam2/sam2.1_hiera_large.pt")
if not sam2_ckpt.exists():
    print("Downloading SAM 2.1 checkpoint...")
    sam2_url = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
    urllib.request.urlretrieve(sam2_url, sam2_ckpt)
    print("✅ SAM 2.1 checkpoint downloaded.")

# 2. Snapshot Grounding DINO Tiny if missing
dino_dir = Path("/models/grounding-dino-tiny")
if not (dino_dir / "config.json").exists():
    print("Downloading Grounding DINO Tiny via transformers...")
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
    detector = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-tiny")
    processor.save_pretrained("/models/grounding-dino-tiny")
    detector.save_pretrained("/models/grounding-dino-tiny")
    print("✅ Grounding DINO Tiny snapshot saved.")

# 3. Download COLMAP Vocab Tree if missing
vocab_tree = Path("/models/vocab_tree_flickr100k.bin")
if not vocab_tree.exists():
    print("Downloading COLMAP Vocab Tree (Flickr100k)...")
    try:
        vocab_url = "https://demuc.de/colmap/vocab_tree_flickr100k.bin"
        urllib.request.urlretrieve(vocab_url, vocab_tree)
        print("✅ Vocab tree downloaded.")
    except Exception as e:
        print(f"⚠️ Vocab tree download optional fallback: {e}")

print("✅ Dependencies and Model Checkpoints ready.")


## Cell 4: Configure Secrets & Worker Environment

In [ ]:
#@title 4. Load Secrets and Configure Environment
import os
import socket
import uuid

# Form parameters for non-secret runtime configs
TASK_QUEUE = "CV_TASK_QUEUE_COLAB" #@param {type:"string"}
MAX_CONCURRENT_ACTIVITIES = 1 #@param {type:"integer"}
STORAGE_AUTH_MODE = "presigned" #@param ["presigned", "static"]
CV_ENV = "dev" #@param ["dev", "prod", "stage"]
SCRATCH_DIR = "/content/scratch" #@param {type:"string"}

os.makedirs(SCRATCH_DIR, exist_ok=True)
os.environ["CV_TASK_QUEUE"] = TASK_QUEUE
os.environ["CV_MAX_CONCURRENT_ACTIVITIES"] = str(MAX_CONCURRENT_ACTIVITIES)
os.environ["STORAGE_AUTH_MODE"] = STORAGE_AUTH_MODE
os.environ["CV_ENV"] = CV_ENV
os.environ["SCRATCH_DIR"] = SCRATCH_DIR

# Set explicit Temporal Worker Identity for observability
worker_id = f"colab-t4-{socket.gethostname()}-{uuid.uuid4().hex[:6]}"
os.environ["TEMPORAL_WORKER_IDENTITY"] = worker_id

# Load credentials securely from Colab Secrets
try:
    from google.colab import userdata
    secret_keys = [
        "TEMPORAL_HOST",
        "TEMPORAL_NAMESPACE",
        "PRESIGN_ENDPOINT",
        "MINIO_ENDPOINT",
        "MINIO_ACCESS_KEY",
        "MINIO_SECRET_KEY",
        "MINIO_SECURE",
        "POSTGRES_DSN",
    ]
    for key in secret_keys:
        val = userdata.get(key)
        if val:
            os.environ[key] = str(val)
except ImportError:
    print("Not running inside Google Colab userdata environment; using ambient environment variables.")
except Exception as e:
    print(f"Note loading Colab secrets: {e}")

print(f"✅ Worker Configuration:")
print(f"  • Task Queue: {os.environ.get('CV_TASK_QUEUE')}")
print(f"  • Concurrency: {os.environ.get('CV_MAX_CONCURRENT_ACTIVITIES')}")
print(f"  • Worker Identity: {os.environ.get('TEMPORAL_WORKER_IDENTITY')}")
print(f"  • Storage Auth Mode: {os.environ.get('STORAGE_AUTH_MODE')}")
print(f"  • Temporal Host: {os.environ.get('TEMPORAL_HOST', 'localhost:7233')}")
print(f"  • Scratch Dir: {os.environ.get('SCRATCH_DIR')}")


## Cell 5: Start Temporal CV Worker

In [ ]:
# 5. Run Temporal Worker Event Loop
import asyncio
import logging
import sys
from common.config import settings
from worker import main

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

print(f"🚀 Starting Temporal Python Worker on task queue: {settings.task_queue}...")
print(f"   Worker Identity: {settings.worker_identity}")
print("   Press Stop Button (⏹) on this cell to halt the worker.")

try:
    await main()
except asyncio.CancelledError:
    print("🛑 Worker stopped cleanly.")
except Exception as e:
    print(f"❌ Worker encountered error: {e}")
    raise


## Cell 6: Liveness & Status Health Check Hook

In [ ]:
# 6. Health & Liveness Hook (Phase 6 / 10 Observability Hook)
import os
import time
from common.config import settings

print(f"Worker Status Check:")
print(f"  - Task Queue: {settings.task_queue}")
print(f"  - Host: {settings.temporal_host}")
print(f"  - Scratch Directory: {settings.scratch_dir}")
print(f"  - Scratch Directory Exists: {os.path.exists(settings.scratch_dir)}")
print(f"  - Time (UTC): {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}")
